# Vascular Alignment Figure

This notebook loads `Table_S5_brain_region_mapping_annotated(1).xlsx` without relying on `openpyxl`, then builds a polished lineage-alignment figure and a compact supporting heatmap.


In [ ]:
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
})
sns.set_theme(style="white")


plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_style("whitegrid")


XLSX_PATH = Path("Table_S5_brain_region_mapping_corrected.xlsx")


def _col_to_idx(col):
    n = 0
    for ch in col:
        if ch.isalpha():
            n = n * 26 + (ord(ch.upper()) - 64)
    return n - 1


def read_xlsx_without_openpyxl(path):
    ns = {
        "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "pr": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with zipfile.ZipFile(path) as z:
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("pr:Relationship", ns)}

        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            shared_root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in shared_root.findall("a:si", ns):
                shared.append("".join(t.text or "" for t in si.findall(".//a:t", ns)))

        sheets = {}
        for sheet in workbook.find("a:sheets", ns):
            name = sheet.attrib["name"]
            rid = sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
            target = rel_map[rid]
            if target.startswith("/"):
                target = target.lstrip("/")
            elif not target.startswith("xl/"):
                target = f"xl/{target}"

            root = ET.fromstring(z.read(target))
            rows = []
            for row in root.findall(".//a:sheetData/a:row", ns):
                vals = {}
                for cell in row.findall("a:c", ns):
                    ref = cell.attrib.get("r", "")
                    match = re.match(r"([A-Z]+)", ref)
                    idx = _col_to_idx(match.group(1)) if match else len(vals)
                    cell_type = cell.attrib.get("t")
                    value = cell.find("a:v", ns)
                    out = ""
                    if cell_type == "s" and value is not None:
                        out = shared[int(value.text)]
                    elif cell_type == "inlineStr":
                        out = "".join(t.text or "" for t in cell.findall(".//a:t", ns))
                    elif value is not None:
                        out = value.text
                    vals[idx] = out
                if vals:
                    max_idx = max(vals)
                    rows.append([vals.get(i, "") for i in range(max_idx + 1)])

            if not rows:
                sheets[name] = pd.DataFrame()
                continue

            header_idx = None
            for i, row in enumerate(rows):
                if any(str(cell).strip() == "Mouse acronym" for cell in row):
                    header_idx = i
                    break
            if header_idx is None:
                header_idx = 0
            header = rows[header_idx]
            data = rows[header_idx + 1:]

            width = max([len(header)] + [len(r) for r in data] if data else [len(header)])
            header = header + [""] * (width - len(header))
            if width > 1 and header[1] == "":
                header[1] = "Mouse region"
            clean_header = []
            seen = {}
            for idx, col in enumerate(header):
                col = str(col).strip()
                if not col:
                    col = f"Unnamed_{idx}"
                seen[col] = seen.get(col, 0) + 1
                if seen[col] > 1:
                    col = f"{col}_{seen[col]}"
                clean_header.append(col)
            data = [r + [""] * (width - len(r)) for r in data]
            sheets[name] = pd.DataFrame(data, columns=clean_header)

    return sheets


In [ ]:
sheets = read_xlsx_without_openpyxl(XLSX_PATH)
mapping_df = sheets["Corrected mapping"].copy()
legend_df = sheets["Legend"].copy()

mapping_df = mapping_df.mask(mapping_df.eq(""))
drop_cols = [c for c in mapping_df.columns if str(c).startswith("Unnamed_") and mapping_df[c].isna().all()]
if drop_cols:
    mapping_df = mapping_df.drop(columns=drop_cols)
mapping_df["Mouse broad embryologic lineage"] = mapping_df["Mouse broad embryologic lineage"].fillna("Unassigned")
mapping_df["Human broad embryologic lineage"] = mapping_df["Human broad embryologic lineage"].fillna("Unassigned")
mapping_df["Mouse cluster"] = mapping_df["Mouse cluster"].fillna("Unassigned")
mapping_df["Human cluster"] = mapping_df["Human cluster"].fillna("Unassigned")
mapping_df["Mouse anatomical division"] = mapping_df["Mouse anatomical division"].fillna("Unassigned")
mapping_df["Human region layer"] = mapping_df["Human region layer"].fillna("Unassigned")

print(mapping_df.shape)
mapping_df.head()


In [ ]:
# Main figure: broad embryologic lineage alignment between mouse and human
LEFT_COL = "Mouse broad embryologic lineage"
RIGHT_COL = "Human broad embryologic lineage"
FIG_OUT = "vascular_alignment_lineage_alluvial.png"

preferred_order = [
    "Telencephalon",
    "Diencephalon",
    "Mesencephalon",
    "Rhombencephalon",
    "Spinal cord",
    "Telencephalon-associated tract/white matter",
    "Non-parenchymal interface",
    "Major vessel compartment",
    "Unassigned",
]

palette = {
    "Telencephalon": "#4E79A7",
    "Diencephalon": "#59A14F",
    "Mesencephalon": "#F28E2B",
    "Rhombencephalon": "#E15759",
    "Spinal cord": "#B07AA1",
    "Telencephalon-associated tract/white matter": "#9C755F",
    "Non-parenchymal interface": "#76B7B2",
    "Major vessel compartment": "#EDC948",
    "Unassigned": "#BAB0AC",
}


def ordered_categories(values):
    observed = pd.Index(values.dropna().unique().tolist())
    ordered = [x for x in preferred_order if x in observed]
    ordered += sorted(x for x in observed if x not in ordered)
    return ordered


def ribbon(ax, x0, x1, y0a, y0b, y1a, y1b, color, alpha=0.58):
    c = 0.22 * (x1 - x0)
    verts = [
        (x0, y0a),
        (x0 + c, y0a),
        (x1 - c, y1a),
        (x1, y1a),
        (x1, y1b),
        (x1 - c, y1b),
        (x0 + c, y0b),
        (x0, y0b),
        (x0, y0a),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))


flow_df = (
    mapping_df.groupby([LEFT_COL, RIGHT_COL], dropna=False)
    .size()
    .reset_index(name="n")
)

left_order = ordered_categories(flow_df[LEFT_COL])
right_order = ordered_categories(flow_df[RIGHT_COL])
left_totals = flow_df.groupby(LEFT_COL)["n"].sum().reindex(left_order)
right_totals = flow_df.groupby(RIGHT_COL)["n"].sum().reindex(right_order)

gap = 1.4
bar_w = 0.06
x_left = 0.14
x_right = 0.86

left_pos = {}
y = 0.0
for cat in left_order:
    left_pos[cat] = y
    y += left_totals[cat] + gap
height_total = y - gap

right_pos = {}
y = 0.0
for cat in right_order:
    right_pos[cat] = y
    y += right_totals[cat] + gap

left_offsets = {cat: 0.0 for cat in left_order}
right_offsets = {cat: 0.0 for cat in right_order}
flow_df = flow_df.sort_values([LEFT_COL, RIGHT_COL], key=lambda s: s.map({v: i for i, v in enumerate(left_order + right_order)}).fillna(999))

fig, ax = plt.subplots(figsize=(15.5, 11.5), facecolor="#f6f1e8")
ax.set_facecolor("#f6f1e8")

for row in flow_df.to_dict("records"):
    src = row[LEFT_COL]
    dst = row[RIGHT_COL]
    val = float(row["n"])
    y0a = left_pos[src] + left_offsets[src]
    y0b = y0a + val
    y1a = right_pos[dst] + right_offsets[dst]
    y1b = y1a + val
    ribbon(ax, x_left + bar_w, x_right - bar_w, y0a, y0b, y1a, y1b, palette.get(src, "#999999"))
    left_offsets[src] += val
    right_offsets[dst] += val

for cat in left_order:
    y0 = left_pos[cat]
    h = left_totals[cat]
    color = palette.get(cat, "#999999")
    ax.add_patch(Rectangle((x_left, y0), bar_w, h, facecolor=color, edgecolor="white", linewidth=1.2))
    ax.text(x_left - 0.018, y0 + h / 2, f"{cat}\n{int(h)}", ha="right", va="center", fontsize=11, color="#1c1c1c")

for cat in right_order:
    y0 = right_pos[cat]
    h = right_totals[cat]
    color = palette.get(cat, "#999999")
    ax.add_patch(Rectangle((x_right - bar_w, y0), bar_w, h, facecolor=color, edgecolor="white", linewidth=1.2))
    ax.text(x_right + 0.018, y0 + h / 2, f"{cat}\n{int(h)}", ha="left", va="center", fontsize=11, color="#1c1c1c")

ax.text(x_left, -3.2, "Mouse broad embryologic lineage", ha="left", va="bottom", fontsize=15, weight="bold", color="#16324f")
ax.text(x_right - bar_w, -3.2, "Human broad embryologic lineage", ha="left", va="bottom", fontsize=15, weight="bold", color="#16324f")
ax.text(0.5, -8.0, "Annotated cross-species region mapping", ha="center", va="bottom", fontsize=21, weight="bold", color="#102a43")
ax.text(0.5, -5.2, "Ribbon width reflects the number of mapped brain-region pairs in Table S5", ha="center", va="bottom", fontsize=12, color="#4a5568")

ax.set_xlim(0, 1)
ax.set_ylim(height_total + 2.0, -10)
ax.axis("off")

plt.savefig(FIG_OUT, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {FIG_OUT}")


In [ ]:
# Supporting view: mouse anatomical division vs human region layer
HEATMAP_OUT = "vascular_alignment_supporting_heatmap.png"

heat = (
    mapping_df.groupby(["Mouse anatomical division", "Human region layer"], dropna=False)
    .size()
    .unstack(fill_value=0)
)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index, heat.sum(axis=0).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12.5, 7.5), facecolor="#fbf8f3")
sns.heatmap(
    heat,
    cmap=sns.light_palette("#1f4e79", as_cmap=True),
    annot=True,
    fmt="d",
    linewidths=0.8,
    linecolor="#f2e9de",
    cbar_kws={"label": "Mapped region pairs"},
    ax=ax,
)
ax.set_title("Structural context of the mapping", fontsize=17, weight="bold", pad=16)
ax.set_xlabel("Human region layer", fontsize=12)
ax.set_ylabel("Mouse anatomical division", fontsize=12)
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.tick_params(axis="y", rotation=0, labelsize=10)
plt.savefig(HEATMAP_OUT, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {HEATMAP_OUT}")


## Cluster-Aligned Region Tiles

This view shows mouse and human regions as square tiles grouped by cluster, with curved connectors for each mapped pair.


In [ ]:
# Region tiles grouped by cluster with cross-species alignment links
TILE_OUT = "vascular_alignment_cluster_tiles.png"

mouse_df = mapping_df[[
    "Mouse acronym",
    "Mouse cluster",
    "Mouse broad embryologic lineage",
    "Mouse anatomical division",
]].drop_duplicates().copy()
mouse_df = mouse_df.rename(columns={
    "Mouse acronym": "region",
    "Mouse cluster": "cluster",
    "Mouse broad embryologic lineage": "lineage",
    "Mouse anatomical division": "division",
})
mouse_df["species"] = "Mouse"
mouse_df = mouse_df.dropna(subset=["region"]).copy()
mouse_df["cluster"] = mouse_df["cluster"].fillna("Unassigned")
mouse_df["lineage"] = mouse_df["lineage"].fillna("Unassigned")
mouse_df["division"] = mouse_df["division"].fillna("Unassigned")

human_label_col = "Human acronym" if "Human acronym" in mapping_df.columns else "Human region"

human_df = mapping_df[[
    human_label_col,
    "Human region",
    "Human cluster",
    "Human broad embryologic lineage",
    "Human region layer",
]].drop_duplicates().copy()
human_df = human_df.rename(columns={
    human_label_col: "region",
    "Human region": "region_full",
    "Human cluster": "cluster",
    "Human broad embryologic lineage": "lineage",
    "Human region layer": "division",
})
human_df["species"] = "Human"
human_df = human_df.dropna(subset=["region"]).copy()
human_df["region_full"] = human_df["region_full"].fillna(human_df["region"])
human_df["cluster"] = human_df["cluster"].fillna("Unassigned")
human_df["lineage"] = human_df["lineage"].fillna("Unassigned")
human_df["division"] = human_df["division"].fillna("Unassigned")

link_df = mapping_df[[
    "Mouse acronym",
    human_label_col,
    "Human region",
    "Mouse broad embryologic lineage",
    "Human broad embryologic lineage",
]].drop_duplicates().copy()
link_df = link_df.rename(columns={human_label_col: "Human label"})
link_df["Human label"] = link_df["Human label"].fillna(link_df["Human region"])
link_df = link_df.dropna(subset=["Mouse acronym", "Human label"]).copy()
link_df["Mouse broad embryologic lineage"] = link_df["Mouse broad embryologic lineage"].fillna("Unassigned")
link_df["Human broad embryologic lineage"] = link_df["Human broad embryologic lineage"].fillna("Unassigned")
link_df = link_df[
    (link_df["Mouse broad embryologic lineage"] != "Unassigned")
    & (link_df["Human broad embryologic lineage"] != "Unassigned")
].copy()

mouse_cluster_order = ["Cluster_1", "Cluster_2", "Cluster_3", "Cluster_4", "Cluster_5", "Unassigned"]
human_cluster_order = ["Cluster_1", "Cluster_3", "Cluster_2", "Cluster_4", "Cluster_5", "Unassigned"]

def order_cluster_values(values, preferred_order):
    observed = pd.Index(values.fillna("Unassigned").unique().tolist())
    ordered = [x for x in preferred_order if x in observed]
    ordered += sorted(x for x in observed if x not in ordered)
    return ordered

TILE_SIZE = 0.82
LINEAGE_MARKER_SIZE = 95
LINEAGE_MARKER_OFFSET_MOUSE = 0.34
LINEAGE_MARKER_OFFSET_HUMAN = 0.34
TILE_GAP = 0.22
CLUSTER_GAP = 1.55
MOUSE_LABEL_OFFSET = 0.95
HUMAN_LABEL_OFFSET = 0.95
CLUSTER_BOX_PAD_X = 0.20
CLUSTER_BOX_PAD_Y = 0.48

base_link_pairs = link_df[["Mouse acronym", "Human label"]].dropna().drop_duplicates()
mouse_to_human = base_link_pairs.groupby("Mouse acronym")["Human label"].apply(list).to_dict()
human_to_mouse = base_link_pairs.groupby("Human label")["Mouse acronym"].apply(list).to_dict()
STEP_X = TILE_SIZE + TILE_GAP

def prepare_layout_df(df, preferred_order):
    df = df.copy()
    df["cluster"] = df["cluster"].fillna("Unassigned")
    cluster_levels = order_cluster_values(df["cluster"], preferred_order)
    df["base_cluster_rank"] = df["cluster"].map({c: i for i, c in enumerate(cluster_levels)})
    df["base_lineage"] = df["lineage"].fillna("Unassigned")
    df["base_division"] = df["division"].fillna("Unassigned")
    df["base_region"] = df["region"].fillna("Unassigned")
    df = df.sort_values(["base_cluster_rank", "base_lineage", "base_division", "base_region"]).reset_index(drop=True)
    df["base_rank"] = np.arange(len(df), dtype=float)
    return df

def compute_cluster_starts(df, preferred_order):
    cluster_levels = order_cluster_values(df["cluster"], preferred_order)
    starts = {}
    x = 0.0
    for cluster in cluster_levels:
        sub = df[df["cluster"] == cluster]
        if sub.empty:
            continue
        starts[cluster] = x
        x += len(sub) * STEP_X + CLUSTER_GAP
    return cluster_levels, starts

def initial_align_sort(sub, partner_x_map, partner_map):
    sub = sub.copy()
    def target_x(region):
        partners = [p for p in partner_map.get(region, []) if p in partner_x_map]
        if partners:
            return float(np.mean([partner_x_map[p] for p in partners]))
        return np.inf
    sub["target_x"] = sub["region"].map(target_x)
    return sub.sort_values(["target_x", "base_lineage", "base_division", "base_region"]).drop(columns="target_x").reset_index(drop=True)

def order_objective(regions, cluster_start, partner_x_map, partner_map):
    total = 0.0
    for idx, region in enumerate(regions):
        x_here = cluster_start + idx * STEP_X
        partners = [p for p in partner_map.get(region, []) if p in partner_x_map]
        if not partners:
            continue
        total += sum(abs(x_here - partner_x_map[p]) for p in partners)
    return total

def optimize_cluster_order(sub, cluster_start, partner_x_map=None, partner_map=None):
    sub = sub.copy().reset_index(drop=True)
    if partner_x_map is None or partner_map is None:
        return sub
    sub = initial_align_sort(sub, partner_x_map, partner_map)
    regions = sub["region"].tolist()
    best_score = order_objective(regions, cluster_start, partner_x_map, partner_map)
    improved = True
    while improved and len(regions) > 1:
        improved = False
        best_swap = None
        for i in range(len(regions) - 1):
            for j in range(i + 1, len(regions)):
                trial = regions.copy()
                trial[i], trial[j] = trial[j], trial[i]
                trial_score = order_objective(trial, cluster_start, partner_x_map, partner_map)
                if trial_score + 1e-9 < best_score:
                    best_score = trial_score
                    best_swap = (i, j, trial)
        if best_swap is not None:
            _, _, regions = best_swap
            improved = True
    order_idx = {region: idx for idx, region in enumerate(regions)}
    sub["order_idx"] = sub["region"].map(order_idx)
    return sub.sort_values("order_idx").drop(columns="order_idx").reset_index(drop=True)

def build_x_map(df, preferred_order):
    cluster_levels, cluster_starts = compute_cluster_starts(df, preferred_order)
    positions = {}
    for cluster in cluster_levels:
        sub = df[df["cluster"] == cluster].reset_index(drop=True)
        if sub.empty:
            continue
        start = cluster_starts[cluster]
        for idx, region in enumerate(sub["region"].tolist()):
            positions[region] = start + idx * STEP_X + TILE_SIZE / 2
    return positions

def optimize_layout_df(df, preferred_order, partner_x_map=None, partner_map=None):
    prepared = prepare_layout_df(df, preferred_order)
    cluster_levels, cluster_starts = compute_cluster_starts(prepared, preferred_order)
    out = []
    for cluster in cluster_levels:
        sub = prepared[prepared["cluster"] == cluster].copy()
        if sub.empty:
            continue
        sub = optimize_cluster_order(sub, cluster_starts[cluster], partner_x_map=partner_x_map, partner_map=partner_map)
        out.append(sub)
    if not out:
        return prepared.iloc[0:0].copy()
    return pd.concat(out, ignore_index=True)

mouse_sorted = optimize_layout_df(mouse_df, mouse_cluster_order)
human_sorted = optimize_layout_df(human_df, human_cluster_order)
for _ in range(6):
    human_x = build_x_map(human_sorted, human_cluster_order)
    mouse_sorted = optimize_layout_df(mouse_df, mouse_cluster_order, partner_x_map=human_x, partner_map=mouse_to_human)
    mouse_x = build_x_map(mouse_sorted, mouse_cluster_order)
    human_sorted = optimize_layout_df(human_df, human_cluster_order, partner_x_map=mouse_x, partner_map=human_to_mouse)

def layout_tiles(df, y0, preferred_order):
    df = df.copy()
    df["cluster"] = df["cluster"].fillna("Unassigned")
    cluster_levels = order_cluster_values(df["cluster"], preferred_order)
    x = 0.0
    positions = {}
    cluster_boxes = []
    for cluster in cluster_levels:
        sub = df[df["cluster"] == cluster].reset_index(drop=True)
        if sub.empty:
            continue
        x_start = x
        for _, row in sub.iterrows():
            positions[row["region"]] = {
                "x": x,
                "y": y0,
                "cluster": cluster,
                "lineage": row["lineage"],
                "division": row["division"],
            }
            x += TILE_SIZE + TILE_GAP
        x_end = x - TILE_GAP
        cluster_boxes.append((cluster, x_start, x_end))
        x += CLUSTER_GAP
    return positions, cluster_boxes, x

mouse_pos, mouse_boxes, mouse_width = layout_tiles(mouse_sorted, y0=7.2, preferred_order=mouse_cluster_order)
human_pos, human_boxes, human_width = layout_tiles(human_sorted, y0=2.6, preferred_order=human_cluster_order)
x_max = max(mouse_width, human_width)

lineage_palette = {
    "Telencephalon": "#4A4A4A",
    "Diencephalon": "#4A4A4A",
    "Mesencephalon": "#4A4A4A",
    "Rhombencephalon": "#4A4A4A",
    "Spinal cord": "#4A4A4A",
    "Telencephalon-associated tract/white matter": "#4A4A4A",
    "Non-parenchymal interface": "#4A4A4A",
    "Major vessel compartment": "#4A4A4A",
    "Unassigned": "#7A7A7A",
}
lineage_shape_map = {
    "Telencephalon": "o",
    "Diencephalon": "s",
    "Mesencephalon": "^",
    "Rhombencephalon": "D",
    "Spinal cord": "P",
    "Telencephalon-associated tract/white matter": "v",
    "Non-parenchymal interface": "X",
    "Major vessel compartment": "*",
    "Unassigned": "h",
}

mouse_division_palette = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Subcortical Region": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
    "Watershed": "#C9A200",
    "Limbic": "#87CEEB",
    "Unassigned": "#c9c9c9",
}

human_layer_palette = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Subcortical Region": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
    "Watershed": "#C9A200",
    "Limbic": "#87CEEB",
    "Unassigned": "#c9c9c9",
}

TILE_OUT_PDF = TILE_OUT.replace('.png', '.pdf')

fig, ax = plt.subplots(figsize=(max(18, 0.34 * x_max), 12.2), facecolor="white")
fig.patch.set_facecolor("white")
fig.patch.set_alpha(1.0)
ax.set_facecolor("white")

for row in link_df.to_dict("records"):
    m = row["Mouse acronym"]
    h = row["Human label"]
    if m not in mouse_pos or h not in human_pos:
        continue
    x0 = mouse_pos[m]["x"] + TILE_SIZE / 2
    y0 = mouse_pos[m]["y"]
    x1 = human_pos[h]["x"] + TILE_SIZE / 2
    y1 = human_pos[h]["y"] + TILE_SIZE
    ctrl_y = (y0 + y1) / 2
    verts = [(x0, y0), (x0, ctrl_y), (x1, ctrl_y), (x1, y1)]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4]
    color = lineage_palette.get(mouse_pos[m]["lineage"], "#999999")
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor="none", edgecolor=color, lw=1.2, alpha=0.23, capstyle="round"))

def draw_tiles(positions, boxes, label_mode):
    if not positions:
        return
    y_min = min(info["y"] for info in positions.values())
    for region, info in positions.items():
        if label_mode == "top":
            color = mouse_division_palette.get(info["division"], "#c9c9c9")
        else:
            color = human_layer_palette.get(info["division"], "#c9c9c9")
        ax.add_patch(Rectangle((info["x"], info["y"]), TILE_SIZE, TILE_SIZE, facecolor=color, edgecolor="white", linewidth=1.4, zorder=3))
        lineage_color = lineage_palette.get(info["lineage"], "#c9c9c9")
        lineage_shape = lineage_shape_map.get(info["lineage"], "o")
        if label_mode == "top":
            ax.scatter(info["x"] + TILE_SIZE / 2, info["y"] + TILE_SIZE + LINEAGE_MARKER_OFFSET_MOUSE, s=LINEAGE_MARKER_SIZE, marker=lineage_shape, facecolor=lineage_color, edgecolor="#243b53", linewidth=0.6, zorder=5)
            ax.text(info["x"] + TILE_SIZE / 2, info["y"] + TILE_SIZE + MOUSE_LABEL_OFFSET, region, rotation=90, ha="center", va="bottom", fontsize=8.0, color="#1c1c1c", bbox=dict(facecolor="white", edgecolor="none", pad=0.25))
        else:
            ax.scatter(info["x"] + TILE_SIZE / 2, info["y"] - LINEAGE_MARKER_OFFSET_HUMAN, s=LINEAGE_MARKER_SIZE, marker=lineage_shape, facecolor=lineage_color, edgecolor="#243b53", linewidth=0.6, zorder=5)
            ax.text(info["x"] + TILE_SIZE / 2, info["y"] - HUMAN_LABEL_OFFSET, region, rotation=90, ha="center", va="top", fontsize=8.0, color="#1c1c1c", bbox=dict(facecolor="white", edgecolor="none", pad=0.25))
    for cluster, x0, x1 in boxes:
        box_y = y_min - CLUSTER_BOX_PAD_Y if label_mode == "top" else y_min - (CLUSTER_BOX_PAD_Y + 0.28)
        ax.add_patch(Rectangle((x0 - CLUSTER_BOX_PAD_X, box_y), (x1 - x0) + 2 * CLUSTER_BOX_PAD_X, TILE_SIZE + 2 * CLUSTER_BOX_PAD_Y + LINEAGE_MARKER_OFFSET_MOUSE, facecolor="none", edgecolor="#8d99ae", linewidth=1.2, zorder=2))
        if label_mode == "top":
            ax.text((x0 + x1) / 2, y_min + TILE_SIZE + 3.05, cluster, ha="center", va="center", fontsize=11, weight="bold", color="#243b53")
        else:
            ax.text((x0 + x1) / 2, y_min - 3.35, cluster, ha="center", va="center", fontsize=11, weight="bold", color="#243b53")

draw_tiles(mouse_pos, mouse_boxes, "top")
draw_tiles(human_pos, human_boxes, "bottom")

#ax.text(x_max / 2, 11.9, "Cross-species region alignment grouped by cluster", ha="center", va="bottom", fontsize=22, weight="bold", color="#102a43")
#ax.text(x_max / 2, 11.4, "Square tiles encode anatomy; lineage is shown by symbol and curves mark mapped pairs", ha="center", va="bottom", fontsize=12, color="#52606d")

legend_items = [x for x in preferred_order if x in set(mouse_df["lineage"]).union(set(human_df["lineage"]))]
legend_handles = [mpl.lines.Line2D([0], [0], marker=lineage_shape_map.get(k, 'o'), linestyle='', markerfacecolor=lineage_palette.get(k, '#999999'), markeredgecolor='#243b53', markeredgewidth=0.6, markersize=9, label=k) for k in legend_items]
fig.subplots_adjust(bottom=0.26)
legend1 = fig.legend(legend_handles, legend_items, title="Embryonic lineage", frameon=False, ncol=3, bbox_to_anchor=(0.5, 0.11), loc="lower center")
layer_items = [x for x in human_df["division"].fillna("Unassigned").drop_duplicates().tolist() if pd.notna(x) and str(x).lower() != 'nan']
layer_handles = [Rectangle((0, 0), 1, 1, facecolor=human_layer_palette.get(k, "#c9c9c9"), edgecolor="none") for k in layer_items]
legend2 = fig.legend(layer_handles, layer_items, title="Region Layer", frameon=False, ncol=4, bbox_to_anchor=(0.5, 0.02), loc="lower center")

ax.set_xlim(-1, x_max + 1)
ax.set_ylim(-1.6, 11.7)
ax.set_aspect('equal', adjustable='box')
ax.axis("off")
plt.savefig(TILE_OUT, dpi=300, bbox_inches="tight", facecolor="white", edgecolor="none", transparent=False)
plt.savefig(TILE_OUT_PDF, bbox_inches="tight", facecolor="white", edgecolor="none", transparent=False)
plt.show()
print(f"Saved: {TILE_OUT}")
print(f"Saved: {TILE_OUT_PDF}")
